<!-- source: new + slide 32–33 + slide 38 -->
# M3 · RAG jako narzędzie: raporty, chunking, AI Search

**Przebieg:** prezentacja, demo wzorca, lab

> *„Mój zespół chce też chatbota, który odpowiada na pytania kontekstowe, nie tylko liczbowe. Czy asystent może czytać nasze raporty i odpowiadać z cytatami?”* — VP of Sales, TechRetail Corp

Tabela odpowiada „ile”, a raport odpowiada „dlaczego” i „co z tym zrobić”. W tym module 10 raportów PDF staje się drugim źródłem wiedzy agenta.

```
OFFLINE (raz)   PDF w Volume → ai_parse_document → tekst per strona → chunki 600/100 → embeddingi → indeks AI Search
ONLINE (pytanie)  pytanie → embedding → top-3 chunki → prompt z fragmentami → odpowiedź z cytatami [raport #fragment]
```

| Część | Co robisz | Lab |
|---|---|---|
| 1 | parsowanie PDF (albo gotowy checkpoint) | wywołanie `ai_parse_document` |
| 2 | chunking i jego parametry | `chunk_size`, `chunk_overlap`, separatory |
| 3 | embedding policzony ręcznie | podobieństwo kosinusowe |
| 4 | indeks AI Search i RAG z cytatami | kontekst i prompt |
| 5 | ANN, HYBRID, FULL_TEXT, filtr; Playground z indeksem | UI |

**Demo wzorca (Krzysztof):** `workshop/pattern/p3_rag_robotics`, czyli cały RAG na raportach o robotyce (3 strony, chunking 2000/200). U nas raporty mają 5 stron i 600/100. Porównaj i zapamiętaj, skąd bierze się ta różnica.

**AI Search** to nowa nazwa **Vector Search** (od czerwca 2026). Klasy w bibliotekach, np. `VectorSearchRetrieverTool`, jeszcze noszą starą nazwę.

**Free Edition:** masz 1 endpoint AI Search. Jeśli nie jest gotowy (`PROVISIONING`), notebook sam przejdzie w **tryb offline**: `retrieve_local()` liczy podobieństwo na przygotowanych embeddingach. Ćwiczenia działają w obu trybach.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
# source: new
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

w = WorkspaceClient()
llm = w.serving_endpoints.get_open_ai_client()
USERNAME = spark.sql("SELECT current_user()").first()[0]
DATA_DIR = Path(os.getcwd()).parent / "data"
SEARCH_COLUMNS = ["chunk_id", "content", "doc_id", "filename", "chunk_position"]

RUN_PARSE = False  # True: ai_parse_document na 10 PDF z Volume (kilka minut). False: gotowy checkpoint.
PARSED_PAGES_PATH = f"{VOLUME_PATH}/parsed_pages"  # obrazy stron zapisywane przez parser (podgląd ramek)
MAX_WAIT_MINUTES = 8  # tyle najwyżej czekamy na endpoint i indeks AI Search, potem tryb offline

assert spark.catalog.tableExists(CHUNKS_TABLE), "Brak tabeli chunków: uruchom najpierw 00_setup."
print(f"Użytkownik: {USERNAME} | dane: {DATA_DIR} | RUN_PARSE={RUN_PARSE}")

<!-- source: WS3[3] + WS3[8] -->
## 1. Od PDF do tekstu: `ai_parse_document`

Analitycy TechRetail przygotowali wcześniej 10 raportów, które leżą w Volume `retail_docs` (skopiował je `00_setup`). `ai_parse_document` to funkcja SQL, która z PDF wyciąga strony, elementy (tytuły, tekst, tabele, wykresy) i ich położenie na stronie (bbox).

Parsowanie 10 plików trwa kilka minut i na Free Edition potrafi trafić na limit (sama funkcja, z obrazami stron i opisami wykresów, działała na Free w testach z lipca 2026). Dlatego domyślnie (`RUN_PARSE = False`) wczytujemy wynik, który prowadzący zapisał tą samą komórką na workspace Premium.

**Poziom 2:** uzupełnij wywołanie funkcji w gałęzi `RUN_PARSE`. Jeśli masz czas i kredyt, ustaw `RUN_PARSE = True` w komórce wyżej i porównaj liczbę znaków z checkpointem.

In [ ]:
# source: WS3[9]
# ZADANIE 6: wywołanie ai_parse_document w SQL.
if RUN_PARSE:
    docs_df = spark.sql(f"""
        WITH parsed_docs AS (
            SELECT
                _metadata.file_name AS filename,
                -- TODO: wywołaj ai_parse_document na kolumnie content z opcjami
                --       MAP('version', '2.0', 'imageOutputPath', '{PARSED_PAGES_PATH}', 'descriptionElementTypes', '*')
                TODO AS parsed
            FROM READ_FILES('{VOLUME_PATH}/', format => 'binaryFile')
            WHERE _metadata.file_name LIKE '%.pdf'
        )
        SELECT
            REPLACE(filename, '.pdf', '') AS doc_id,
            filename,
            concat_ws('\\n\\n', transform(
                try_cast(parsed:document:elements AS ARRAY<VARIANT>),
                element -> try_cast(element:content AS STRING)
            )) AS content,
            to_json(parsed) AS parsed_json
        FROM parsed_docs
        WHERE is_variant_null(parsed:error_status)
    """)
    print("Parsowanie na żywo: ai_parse_document")
else:
    docs_df = spark.createDataFrame(pd.read_parquet(DATA_DIR / "checkpoints" / "retail_rag_docs.parquet"))
    print("(tryb offline — z przygotowanych plików: data/checkpoints/retail_rag_docs.parquet)")

docs_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(DOCS_TABLE)
display(spark.table(DOCS_TABLE).select("doc_id", "filename", F.length("content").alias("znaków")).orderBy("doc_id"))

In [ ]:
# source: WS3[11]
# Co zwrócił parser: strony, elementy i ich typy. Wymaga RUN_PARSE = True (checkpoint nie ma parsed_json).
if "parsed_json" not in spark.table(DOCS_TABLE).columns:
    print("Pominięte: metadane parsowania są tylko po RUN_PARSE = True.")
else:
    display(spark.sql(f"""
        SELECT value:type::string AS typ_elementu, COUNT(*) AS n
        FROM (SELECT parse_json(parsed_json) AS parsed FROM {DOCS_TABLE}), LATERAL variant_explode(parsed:document:elements)
        GROUP BY 1 ORDER BY n DESC
    """))

In [ ]:
# source: WS3[12] + K:Warsztaty_Krzysztof/rag_agent/notebooks/includes/document_renderer.py
# Podgląd strony PDF z ramkami elementów rozpoznanych przez parser. Wymaga RUN_PARSE = True.
import base64
import html
import json
from collections import Counter

TYPE_COLORS = {"title": "#7c3aed", "section_header": "#2563eb", "text": "#16a34a", "table": "#ea580c",
               "figure": "#db2777", "caption": "#0891b2", "page_header": "#6b7280", "page_footer": "#6b7280"}


def render_parsed_page(parsed_json: str, page_no: int = 1, max_width: int = 720) -> None:
    doc = json.loads(parsed_json).get("document", {})
    page = next((p for p in doc.get("pages", []) if int(p.get("id", -1)) == page_no - 1), None)
    image_uri = (page or {}).get("image_uri") or ""
    if not os.path.isfile(image_uri):
        displayHTML(f"<p>Brak obrazu strony {page_no}: <code>{html.escape(image_uri)}</code></p>")
        return
    boxes = [(el, bb["coord"][:4]) for el in doc.get("elements", []) for bb in (el.get("bbox") or [])[:1]
             if bb.get("page_id") == page_no - 1 and len(bb.get("coord") or []) >= 4]
    coords = [c for _, box in boxes for c in box]
    width = max([c for i, c in enumerate(coords) if i % 4 in (0, 2)] or [1000])
    height = max([c for i, c in enumerate(coords) if i % 4 in (1, 3)] or [1400])
    shapes = "".join(
        f'<g><title>{html.escape(str(el.get("type")) + ": " + str(el.get("content") or el.get("description") or "")[:300])}</title>'
        f'<rect x="{min(l, r)}" y="{min(t, b)}" width="{abs(r - l)}" height="{abs(b - t)}" fill="{TYPE_COLORS.get(el.get("type"), "#64748b")}" '
        f'fill-opacity="0.12" stroke="{TYPE_COLORS.get(el.get("type"), "#64748b")}" stroke-width="{max(width, height) * 0.0025}"/></g>'
        for el, (l, t, r, b) in boxes
    )
    legend = ", ".join(f"{kind}: {count}" for kind, count in sorted(Counter(str(el.get("type")) for el, _ in boxes).items()))
    with open(image_uri, "rb") as image:
        image_b64 = base64.b64encode(image.read()).decode("ascii")
    displayHTML(f"""<div style="max-width:{max_width}px;font-family:Arial">
      <p><b>Strona {page_no}</b> · {legend} · najedź na ramkę, żeby zobaczyć treść</p>
      <div style="position:relative"><img src="data:image/png;base64,{image_b64}" style="width:100%;display:block">
      <svg viewBox="0 0 {width} {height}" preserveAspectRatio="none" style="position:absolute;inset:0;width:100%;height:100%">{shapes}</svg></div></div>""")


if "parsed_json" not in spark.table(DOCS_TABLE).columns:
    print("Pominięte: podgląd ramek wymaga RUN_PARSE = True.")
else:
    sample = spark.table(DOCS_TABLE).orderBy("doc_id").first()
    print(sample["filename"])
    render_parsed_page(sample["parsed_json"], page_no=1)
    render_parsed_page(sample["parsed_json"], page_no=2)

<!-- source: WS3[13] -->
## 2. Chunking: dlaczego nie cały dokument

Gdyby indeks miał 10 wierszy (10 całych raportów):
- embedding pięciostronicowego raportu to „średnia” całej treści, więc pytanie o retencję VIP pasuje równie słabo do każdego dokumentu;
- do promptu trafia cały raport: za dużo szumu albo tekst ucięty w połowie tabeli;
- na 10 wierszach filtry i wyszukiwanie hybrydowe nie mają czego przeszukiwać.

Dlatego tniemy tekst na fragmenty po ok. **600 znaków z nakładką 100**. Separator `== page ==` stoi na liście pierwszy, więc splitter najchętniej tnie na granicy strony. Nasze raporty mają 5 stron, co daje 5–8 fragmentów na raport. Przy `chunk_size=2000` wyszłyby po 2, za mało, żeby wyszukiwanie miało z czego wybierać. Od góry ogranicza nas okno modelu embeddingowego (rzędu 512 tokenów): nadmiar jest ucinany bez ostrzeżenia.

Komórka poniżej zamienia wynik parsera na **tekst per strona**. W WS3 zmienne `markdown_df` i `TEXT_COLUMN_TO_CHUNK` ustawiała dopiero opcjonalna komórka, więc chunking bez niej kończył się `NameError`. Tu jest to naprawione.

In [ ]:
# source: WS3[14]
import html as _html
import json
import re

from pyspark.sql.types import StringType


def html_to_plain_text(value: str) -> str:
    """Tabele przychodzą jako HTML: usuwamy tagi, zachowując podziały wierszy."""
    with_breaks = re.sub(r"</(p|div|tr|li|h[1-6])>", "\n", value, flags=re.IGNORECASE)
    no_tags = re.sub(r"<[^>]+>", " ", with_breaks)
    return re.sub(r"[ \t]+", " ", _html.unescape(no_tags)).strip()


def parsed_json_to_plain_text(parsed_json: str) -> str:
    """Spłaszcza elementy do tekstu per strona; strony rozdziela == page ==."""
    doc = json.loads(parsed_json).get("document") or {}
    pages = doc.get("pages") or []
    by_page = {int(p.get("id", i)): [] for i, p in enumerate(pages)}
    for element in sorted(doc.get("elements") or [], key=lambda e: e.get("id", 0)):
        content = element.get("content") or element.get("description")
        text = html_to_plain_text(str(content)) if content else ""
        if text:
            bbox = element.get("bbox") or []
            by_page.setdefault(int(bbox[0].get("page_id", 0)) if bbox else 0, []).append(text)
    order = [int(p.get("id", i)) for i, p in enumerate(pages)] or sorted(by_page)
    return "\n== page ==\n".join("\n".join(by_page.get(page_id, [])) for page_id in order)


docs = spark.table(DOCS_TABLE)
if "plain_text" in docs.columns:
    plain_text_df = docs
else:
    plain_text_df = docs.withColumn("plain_text", F.udf(parsed_json_to_plain_text, StringType())(F.col("parsed_json")))

# Fix względem WS3: zmienne potrzebne w chunkingu ustawiamy tutaj, a nie w opcjonalnej komórce.
markdown_df = plain_text_df
TEXT_COLUMN_TO_CHUNK = "plain_text"
print(plain_text_df.orderBy("doc_id").first()[TEXT_COLUMN_TO_CHUNK][:1200])

<!-- source: WS3[13] + slide 37 -->
**Lab:** ustaw parametry splittera i sprawdź, ile fragmentów powstaje. Spróbuj trzech wariantów: 600/100, 2000/0 i 300/50. Zapisz, jak zmienia się liczba i średnia długość fragmentów.

Twój eksperyment zostaje w pamięci notebooka. Indeks AI Search korzysta z tabeli `retail_rag_chunks` z `00_setup` (600/100), więc nic nie zepsujesz.

In [ ]:
# source: WS3[16]
# ZADANIE 7: parametry chunkingu.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: rozmiar fragmentu w znakach (zacznij od 600) i nakładka (zacznij od 100)
CHUNK_SIZE = ...
CHUNK_OVERLAP = ...
# TODO: separatory od najważniejszego; granica strony "\n== page ==\n" powinna być pierwsza,
#       potem "== page ==", akapit "\n\n", linia "\n", spacja " " i "" na końcu
SEPARATORS = ...

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=SEPARATORS, keep_separator=True
)
source_docs = markdown_df.select("doc_id", "filename", TEXT_COLUMN_TO_CHUNK).orderBy("doc_id").toPandas()
lab_chunks = pd.DataFrame([
    {"doc_id": row["doc_id"], "chunk_position": position, "content": chunk}
    for _, row in source_docs.iterrows()
    for position, chunk in enumerate(splitter.split_text(row[TEXT_COLUMN_TO_CHUNK] or ""))
])

print(f"Parametry {CHUNK_SIZE}/{CHUNK_OVERLAP}: {len(lab_chunks)} fragmentów "
      f"(tabela indeksu {CHUNKS_TABLE}: {spark.table(CHUNKS_TABLE).count()} fragmentów przy 600/100)")
display(lab_chunks.assign(znaków=lab_chunks["content"].str.len())
        .groupby("doc_id").agg(fragmentów=("content", "size"), średnio_znaków=("znaków", "mean")).round(0).reset_index())

<!-- source: WS3[17] + slide 36 -->
## 3. Embedding: tekst zamieniony na 1024 liczby

Podobne znaczenie daje bliskie wektory. Miarą bliskości jest **podobieństwo kosinusowe**: iloczyn skalarny wektorów znormalizowanych do długości 1. Wartość bliska 1 oznacza to samo znaczenie, a wartość bliska 0 inne.

Ten sam model (`databricks-gte-large-en`) koduje raz każdy fragment (offline, robi to indeks) i raz każde pytanie (online). Gdyby po obu stronach były różne modele, wektory nie leżałyby w tej samej przestrzeni.

**Poziom 2:** policz macierz podobieństwa czterech zdań.

In [ ]:
# source: WS3[18]
# ZADANIE 8: podobieństwo kosinusowe.
import time

import openai
from databricks.sdk import WorkspaceClient

# Klient zgodny z OpenAI, jak w M1. Limit zapytań do modeli pay-per-token jest wspólny dla całego
# workspace, więc przy 429 czekamy coraz dłużej i mówimy o tym, zamiast przerywać albo wisieć bez słowa.
embedding_client = WorkspaceClient().serving_endpoints.get_open_ai_client().with_options(max_retries=0, timeout=30)


def embed(texts: list) -> np.ndarray:
    for wait in (5, 10, 20, 30, None):
        try:
            data = embedding_client.embeddings.create(model=EMBEDDING_ENDPOINT, input=texts).data
            return np.array([row.embedding for row in data])
        except openai.RateLimitError:
            if wait is None:
                raise
            print(f"   limit zapytań {EMBEDDING_ENDPOINT} (cała sala naraz), czekam {wait} s…")
            time.sleep(wait)


sentences = [
    "Ile mamy klientów VIP?",
    "Liczba klientów w segmencie 3",                 # to samo znaczenie, inne słowa
    "Jaki stan ma najwięcej klientów?",              # inny temat z tej samej domeny
    "Jaki jest dobry przepis na zupę pomidorową?",   # spoza domeny
]
vectors = embed(sentences)
print(f"Wymiar wektora: {vectors.shape[1]} | pierwsze liczby: {np.round(vectors[0][:5], 4)}")

# TODO 1: znormalizuj każdy wiersz do długości 1 (np.linalg.norm(..., axis=1, keepdims=True))
normalized = ...
# TODO 2: macierz podobieństwa kosinusowego = iloczyn macierzy znormalizowanej i jej transpozycji (@)
similarity = ...

print("\nPodobieństwo do „Ile mamy klientów VIP?”:")
for sentence, score in zip(sentences[1:], similarity[0][1:]):
    print(f"  {score:.3f}  ← {sentence}")
# Oczekiwanie: parafraza najbliżej, zupa najdalej.

<!-- source: WS3[8] + slide 39 -->
## 4. Indeks AI Search (dawniej Vector Search)

| Element | Co to jest | U nas |
|---|---|---|
| **Endpoint** | moc obliczeniowa wyszukiwarki | `retail_rag_search`, uruchomiony w `00_setup` |
| **Delta Sync index** | indeks nad tabelą Delta, sam nadąża za zmianami (Change Data Feed) | `retail_rag_chunks_index` na `retail_rag_chunks` |
| **Managed embeddings** | indeks sam liczy wektory kolumny tekstowej | `content` przez `databricks-gte-large-en` |
| **columns_to_sync** | metadane do cytatów i filtrów | `doc_id`, `filename`, `chunk_position` |

Pierwszy indeks na Free Edition przez kilka minut pokazuje `PROVISIONING_ENDPOINT`, a `sync()` zaraz po utworzeniu zwraca „index is not ready”. To normalne: wystarczy uruchomić komórkę ponownie.

Komórka czeka na endpoint i indeks najwyżej `MAX_WAIT_MINUTES` minut. Jeśli nie zdążą, ustawia `SEARCH_READY = False` i dalsza część działa w trybie offline. Uruchom ją ponownie po przerwie, a zwykle będzie już `True`.

In [ ]:
# source: WS3[19]
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient(disable_notice=True)


def wait_for(check, label: str, minutes: float = MAX_WAIT_MINUTES, every: int = 20) -> bool:
    deadline = time.time() + minutes * 60
    while True:
        ready, status = check()
        print(f"   {label}: {status}")
        if ready:
            return True
        if time.time() > deadline:
            return False
        time.sleep(every)


def endpoint_check():
    state = search_client.get_endpoint(SEARCH_ENDPOINT).get("endpoint_status", {}).get("state", "UNKNOWN")
    return state == "ONLINE", state


SEARCH_READY = False
try:
    if not search_client.endpoint_exists(SEARCH_ENDPOINT):
        print(f"Tworzę endpoint {SEARCH_ENDPOINT} (nie powstał w 00_setup)")
        search_client.create_endpoint(name=SEARCH_ENDPOINT, endpoint_type="STANDARD")
    if wait_for(endpoint_check, "endpoint"):
        index_existed = search_client.index_exists(SEARCH_ENDPOINT, SEARCH_INDEX)
        if not index_existed:
            print(f"Tworzę indeks {SEARCH_INDEX} na {CHUNKS_TABLE}")
            search_client.create_delta_sync_index(
                endpoint_name=SEARCH_ENDPOINT,
                index_name=SEARCH_INDEX,
                primary_key="chunk_id",
                source_table_name=CHUNKS_TABLE,
                pipeline_type="TRIGGERED",
                embedding_source_column="content",
                embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
                columns_to_sync=["doc_id", "filename", "chunk_position"],
            )
        index = search_client.get_index(SEARCH_ENDPOINT, SEARCH_INDEX)

        def index_check():
            status = index.describe().get("status", {})
            state = str(status.get("detailed_state", ""))
            ready = bool(status.get("ready")) or state.upper().startswith("ONLINE")
            return ready, f"{state or 'ready=' + str(status.get('ready'))} wierszy={status.get('indexed_row_count')}"

        SEARCH_READY = wait_for(index_check, "indeks")
        if SEARCH_READY and index_existed:
            try:
                index.sync()  # TRIGGERED: dociąga zmiany z tabeli chunków
            except Exception as sync_error:
                print(f"   sync pominięty: {str(sync_error)[:120]}")
except Exception as e:
    print(f"AI Search niedostępny: {type(e).__name__}: {str(e)[:200]}")

print(f"\nSEARCH_READY = {SEARCH_READY}" + ("" if SEARCH_READY else "  →  (tryb offline — z przygotowanych plików)"))

In [ ]:
# source: WS3[20] + WS3[18]
import re

_local_index = None


def rows_from_result(result: dict) -> list:
    """Wynik similarity_search (manifest + data_array) → lista słowników; ostatnia kolumna to score."""
    columns = [column["name"] for column in result["manifest"]["columns"]]
    return [dict(zip(columns, row)) for row in result.get("result", {}).get("data_array", []) or []]


def retrieve_search(question: str, k: int = 3, query_type: str = "ANN", filters: dict | None = None) -> list:
    index = search_client.get_index(SEARCH_ENDPOINT, SEARCH_INDEX)
    result = index.similarity_search(
        query_text=question, columns=SEARCH_COLUMNS, num_results=k, query_type=query_type, filters=filters
    )
    return [{**row, "score": float(row.get("score", 0.0))} for row in rows_from_result(result)]


def retrieve_local(question: str, k: int = 3, query_type: str = "ANN", filters: dict | None = None) -> list:
    """Tryb offline: te same chunki i embeddingi co w indeksie, podobieństwo liczone w numpy."""
    global _local_index
    if _local_index is None:
        chunks = pd.read_parquet(DATA_DIR / "checkpoints" / "retail_rag_chunks.parquet")
        vectors = pd.read_parquet(DATA_DIR / "checkpoints" / "retail_rag_chunk_embeddings.parquet")
        merged = chunks.merge(vectors, on="chunk_id").reset_index(drop=True)
        matrix = np.vstack(merged["embedding"].to_numpy()).astype("float32")
        _local_index = (merged, matrix / np.linalg.norm(matrix, axis=1, keepdims=True))
    merged, matrix = _local_index

    words = [word for word in re.findall(r"\w+", question.lower()) if len(word) > 2]
    keyword = merged["content"].str.lower().apply(lambda text: sum(word in text for word in words)).to_numpy(dtype="float32")
    keyword = keyword / keyword.max() if keyword.max() > 0 else keyword
    if query_type == "FULL_TEXT":
        scores = keyword
    else:
        query = embed([question])[0]
        semantic = matrix @ (query / np.linalg.norm(query))
        scores = semantic if query_type == "ANN" else 0.5 * semantic + 0.5 * keyword  # HYBRID: przybliżenie offline

    candidates = merged.assign(score=scores)
    for column, value in (filters or {}).items():
        candidates = candidates[candidates[column] == value]
    top = candidates.sort_values("score", ascending=False).head(k)
    return [
        {"chunk_id": r.chunk_id, "content": r.content, "doc_id": r.doc_id, "filename": r.filename,
         "chunk_position": int(r.chunk_position), "score": float(r.score)}
        for r in top.itertuples()
    ]


def retrieve(question: str, k: int = 3, query_type: str = "ANN", filters: dict | None = None) -> list:
    if SEARCH_READY:
        return retrieve_search(question, k=k, query_type=query_type, filters=filters)
    return retrieve_local(question, k=k, query_type=query_type, filters=filters)


for chunk in retrieve("Co raporty mówią o retencji klientów VIP?"):
    print(f"[{chunk['doc_id']} #{chunk['chunk_position']}] score={chunk['score']:.3f}  {chunk['content'][:90]!r}")
print("\nTryb:", "AI Search" if SEARCH_READY else "(tryb offline — z przygotowanych plików)")

<!-- source: slide 40 + WS3[20] -->
### RAG z cytatami: szukaj, doklej, odpowiedz

Trzy pytania kontrolne do każdej odpowiedzi RAG:
1. **Ugruntowanie.** Czy odpowiedź wynika z pobranych fragmentów, czy model dopowiedział z pamięci?
2. **Cytaty.** Czy każda liczba i teza ma źródło `[raport #fragment]`?
3. **Uczciwość.** Gdy kontekst nie zawiera odpowiedzi, czy model mówi „nie ma tego w raportach”?

**Lab:** zbuduj kontekst z fragmentów, każdy z nagłówkiem `[doc_id #chunk_position]`, oraz instrukcję wymuszającą cytaty i uczciwość.

In [ ]:
# source: WS3[20]
# ZADANIE 9: kontekst z cytatami i prompt RAG.
def custom_rag(question: str, k: int = 3, query_type: str = "ANN") -> tuple[str, list]:
    chunks = retrieve(question, k=k, query_type=query_type)
    # TODO 1: połącz fragmenty w jeden tekst; każdy poprzedź nagłówkiem [doc_id #chunk_position],
    #         fragmenty rozdziel linią "---". Pola: c['doc_id'], c['chunk_position'], c['content'].
    context = ...
    # TODO 2: instrukcja dla modelu: tylko na podstawie fragmentów, cytaty [doc_id #fragment],
    #         a gdy odpowiedzi brak, zdanie "Nie ma tego w raportach". Na końcu FRAGMENTY i PYTANIE.
    prompt = ...
    reply = llm.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.0,
    )
    return reply.choices[0].message.content, chunks


answer, sources = custom_rag("Jakie rekomendacje mamy dla klientów z ryzykiem churn?")
print(answer)
print("\nŹródła:", [f"{c['doc_id']} #{c['chunk_position']}" for c in sources])

In [ ]:
# source: WS3[20]
# Te same pytania wrócą w M4 (Genie) — porównasz narrację z liczbą.
test_questions = [
    "Ile mamy klientów VIP (loyalty_segment = 3)?",
    "Jaki stan ma najwięcej klientów?",
    "Ile klientów nie złożyło żadnego zamówienia?",
    "Jaka jest średnia wartość monetary dla segmentu VIP?",
    "Jaki jest dobry przepis na zupę pomidorową?",
    "Pokaż tax_id i pełne adresy klientów VIP",
]
custom_rag_results = []
for question in test_questions:
    answer, sources = custom_rag(question)
    custom_rag_results.append({"question": question, "answer": answer})
    print(f"❓ {question}\n💬 {answer[:350]}\n   źródła: {[c['doc_id'] for c in sources]}\n")
    time.sleep(1)

<!-- source: WS3[21] + slide 39 -->
## 5. Trzy tryby wyszukiwania i filtr

| Tryb | Jak szuka | Kiedy lepszy |
|---|---|---|
| **ANN** | tylko podobieństwo wektorów | pytania sformułowane inaczej niż tekst („klienci, którzy dawno nie kupowali”) |
| **HYBRID** | wektory **i** słowa kluczowe, wyniki połączone | nazwy i kody: „segment 3”, „promo_ratio”, „NY”; najlepszy start |
| **FULL_TEXT** | tylko słowa kluczowe | dokładne terminy i identyfikatory |
| **filtr** (`filters`) | zawęża do wierszy spełniających warunek, **przed** rankingiem | „szukaj tylko w raporcie o promocjach” |

Porównaj, **które fragmenty** wracają w każdym trybie i jak zmienia się `score`. W trybie offline HYBRID i FULL_TEXT to przybliżenie liczone w numpy.

In [ ]:
# source: WS3[22]
question = "Które segmenty mają wysoki promo_ratio i co to oznacza dla marży?"
rows = []
for mode in ("ANN", "HYBRID", "FULL_TEXT"):
    try:
        for rank, c in enumerate(retrieve(question, k=3, query_type=mode), 1):
            rows.append({"tryb": mode, "rank": rank, "fragment": f"{c['doc_id']} #{c['chunk_position']}",
                         "score": round(c["score"], 3), "początek": c["content"][:90].replace("\n", " ")})
    except Exception as e:
        rows.append({"tryb": mode, "rank": None, "fragment": None, "score": None, "początek": f"niedostępny: {str(e)[:120]}"})
display(pd.DataFrame(rows))

print("Filtr doc_id = '08_wskazniki_promocyjne' (tryb ANN):")
for c in retrieve(question, k=3, filters={"doc_id": "08_wskazniki_promocyjne"}):
    print(f"   [{c['doc_id']} #{c['chunk_position']}] score={c['score']:.3f}  {c['content'][:80]!r}")

In [ ]:
# source: WS3[24]
# Reranking: drugi model ocenia trafność top-K i zmienia kolejność. Wymaga AI Search i włączonego rerankera.
# Na Free Edition (testy 07.2026) reranker był zablokowany konfiguracją workspace — spodziewaj się komunikatu.
if not SEARCH_READY:
    print("Pominięte w trybie offline.")
else:
    try:
        from databricks.ai_search.reranker import DatabricksReranker

        index = search_client.get_index(SEARCH_ENDPOINT, SEARCH_INDEX)
        kwargs = dict(query_text=question, columns=["doc_id", "chunk_position", "content"], num_results=5, query_type="HYBRID")
        before = rows_from_result(index.similarity_search(**kwargs))
        after = rows_from_result(index.similarity_search(**kwargs, reranker=DatabricksReranker(columns_to_rerank=["content"])))
        for i, (b, a) in enumerate(zip(before, after), 1):
            print(f"{i}. przed: {b['doc_id']} #{b['chunk_position']:<4} po: {a['doc_id']} #{a['chunk_position']}")
    except Exception as e:
        print(f"Reranker niedostępny w tym workspace: {type(e).__name__}: {str(e)[:200]}")

<!-- source: WS3[25] -->
## 6. Lab w AI Playground: indeks jako Tool

1. **Playground** → model `databricks-meta-llama-3-3-70b-instruct` → wklej `SYSTEM_PROMPT`.
2. **Tools → Add tool → AI Search index** (w części UI jeszcze *Vector Search*) → `workspace.default.retail_rag_chunks_index`.
3. Zapytaj: *„Co raporty mówią o retencji klientów VIP?”* i rozwiń panel narzędzia: zobaczysz pobrane fragmenty (`doc_id`, treść) i odpowiedź z cytatami.
4. Dodaj też obie funkcje z M2. Zapytaj: *„Jaka jest średnia wartość klienta VIP?”*. Które narzędzie wybrał model: funkcję czy indeks? Dlaczego?

Jeśli indeksu nie ma na liście albo `SEARCH_READY = False`, obejrzyj ten krok na ekranie prowadzącego. Wrócisz do niego po przerwie, gdy endpoint będzie `ONLINE`.

<!-- source: WS3[26] -->
### Ten sam RAG jako łańcuch LangChain, ze śladem w MLflow

`custom_rag()` żyje tylko w tym notebooku. Łańcuch LangChain to ten sam pipeline jako obiekt, który można zalogować, wersjonować i wdrożyć. `mlflow.langchain.autolog()` zapisuje każde wywołanie jako **trace**: retriever → fragmenty → prompt → model.

**Gdzie obejrzeć:** *Experiments → sqlday_retail_agent → Traces*. Rozwiń span retrievera (lista fragmentów) i span modelu (pełny prompt).

In [ ]:
# source: WS3[27]
import mlflow
from operator import itemgetter

from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

mlflow.set_experiment(f"/Users/{USERNAME}/{EXPERIMENT_NAME}")
mlflow.langchain.autolog()

RAG_PROMPT = (
    SYSTEM_PROMPT + "\n\nOdpowiadaj wyłącznie na podstawie fragmentów raportów. Cytuj źródła jako [doc_id #fragment].\n\n"
    "Fragmenty raportów:\n{context}\n\nPytanie: {question}\nOdpowiedź:"
)

if SEARCH_READY:
    retriever = DatabricksVectorSearch(
        endpoint=SEARCH_ENDPOINT, index_name=SEARCH_INDEX, columns=SEARCH_COLUMNS
    ).as_retriever(search_kwargs={"k": 3, "query_type": "HYBRID"})
else:
    retriever = RunnableLambda(lambda q: [Document(page_content=c["content"], metadata=c) for c in retrieve_local(q, k=3)])


def latest_question(messages: list) -> str:
    return messages[-1]["content"]


def format_docs(docs: list) -> str:
    return "\n\n".join(f"[{d.metadata.get('doc_id')} #{d.metadata.get('chunk_position')}] {d.page_content}" for d in docs) or "Brak fragmentów."


rag_chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(latest_question),
        "context": itemgetter("messages") | RunnableLambda(latest_question) | retriever | RunnableLambda(format_docs),
    }
    | PromptTemplate.from_template(RAG_PROMPT)
    | ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0, max_tokens=400)
    | StrOutputParser()
)

print(rag_chain.invoke({"messages": [{"role": "user", "content": "Co raporty mówią o klientach z ryzykiem churn?"}]}))
print(f"\nTrace: {mlflow.get_last_active_trace_id()} → Experiments → {EXPERIMENT_NAME} → Traces")

<!-- source: WS3[29] + K:Warsztaty_Krzysztof/rag_agent/notebooks/05_building_assistant.py -->
> **Demo prowadzącego (Premium).** Knowledge Assistant (Agent Bricks) to ten sam RAG bez kodu: wskazujesz Volume z raportami, a on sam tnie, indeksuje i cytuje. Ekspert może poprawiać odpowiedzi przez **Examples i Guidelines** bez zmiany dokumentów. Na Free Edition jest niedostępny, dlatego pokazuje go prowadzący. Porównaj jego odpowiedzi na 6 pytań z Twoim `custom_rag`.


<!-- source: WS3[33] -->
> **Demo prowadzącego (Premium).** Prowadzący odpytuje Knowledge Assistant tymi samymi 6 pytaniami przez klienta zgodnego z OpenAI (Responses API).


<!-- source: new + K:Warsztaty_Krzysztof/rag_agent/notebooks/03_vector_search.py -->
## Poziomy 2 i 3: kiedy skończysz ścieżkę

| Poziom | Zadanie |
|---|---|
| **2. Transfer** | Komórka poniżej: RAG z cytatami na opiniach klientów Bakehouse. Opinie są krótkie, więc **nie tniemy ich na fragmenty**. To też decyzja zależna od dokumentów. Uruchom najpierw komórkę z `embed` (część 3). |
| **3. Wyzwanie** | Raporty TechRetail pocięte inaczej (np. 300/50 i 1500/150) → 3 warianty chunkingu → indeks na endpoincie z M3 → ANN vs HYBRID na 5 pytaniach z oceną trafności (trafił / nie trafił). |

In [ ]:
# source: new + WS3[18] + WS3[20]
import time

# POZIOM 2: ten sam RAG na opiniach klientów sieci piekarni Bakehouse.
reviews = (spark.table("samples.bakehouse.media_customer_reviews")
           .selectExpr("CAST(franchiseID AS STRING) AS source", "review AS content")
           .where("length(review) > 20").limit(60).toPandas())
# TODO 1: embeddingi opinii w paczkach po 8 (funkcja embed z części 3), z time.sleep(1) między paczkami
#         (większa paczka długich tekstów dostaje 429), potem normalizacja wierszy do długości 1
review_vectors = ...


def bakehouse_rag(question: str, k: int = 3) -> str:
    query = embed([question])[0]
    # TODO 2: k najbardziej podobnych opinii (iloczyn skalarny ze znormalizowanym pytaniem, nlargest)
    top = ...
    context = "\n\n".join(f"[franczyza {r.source}] {r.content}" for r in top.itertuples())
    # TODO 3: prompt: tylko z opinii, cytat [franczyza …], uczciwe „Nie ma tego w opiniach”
    prompt = ...
    reply = llm.chat.completions.create(model=LLM_ENDPOINT, messages=[{"role": "user", "content": prompt}], max_tokens=300, temperature=0.0)
    return reply.choices[0].message.content


print(bakehouse_rag("Co klienci chwalą, a na co się skarżą?"))

<!-- source: new + slide 38 + K:Warsztaty_Krzysztof/rag_agent/notebooks/02_chunking.py -->
## Karta wzorca: dokumenty jako narzędzie

1. **Dokumenty → tekst** (`ai_parse_document`; wykresy dostają opis).
2. **Rozmiar fragmentu = długość typowej sekcji** Twoich dokumentów; krótkich tekstów (opinie, notatki) nie tnij wcale.
3. **Tabela fragmentów** z kluczem i metadanymi do cytatów i filtrów, z włączonym Change Data Feed.
4. **Indeks** (zacznij od HYBRID) albo, na start, wyszukiwanie po słowach kluczowych z tym samym kontraktem.
5. **Odpowiedź tylko z fragmentów, z cytatem**, a gdy odpowiedzi brak, uczciwe „nie ma tego w dokumentach”.

**Canvas agenta** (`workshop/transfer/canvas_agenta.md`): jakie masz dokumenty, jak długa jest typowa sekcja, jakie metadane posłużą do filtrów.

<!-- source: new + WS3[39] -->
## Podsumowanie

- RAG to **źródło wiedzy, nie decydent**. Sam nie wie, czy pytanie dotyczy dokumentów czy tabeli, dlatego w M5 będzie jednym z narzędzi agenta.
- O jakości decyduje kontekst: 3 trafne fragmenty są lepsze niż 3 całe raporty. Chunking, tryb wyszukiwania i filtry to pokrętła, które masz tylko we własnym RAG.
- Embedding to 1024 liczby, a podobieństwo kosinusowe to iloczyn skalarny znormalizowanych wektorów. Indeks robi to samo dla każdego pytania.
- Odpowiedź RAG ma być ugruntowana, cytowana i uczciwa. Raport jest jednak **snapshotem**: liczbę „na dziś” daje tabela. To różnica, którą agent musi znać (M4).

**Dalej:** M4. Te same pytania zadasz Genie Agentowi nad tabelą, a potem zabezpieczysz dane maską i filtrem.